In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq


os.environ["groq_api_key"] = os.getenv("groq_api_key")

model = ChatGroq(model="llama-3.3-70B-Versatile")
for chunk in model.stream("hi who are you?"):
    print(chunk.text, end="" , flush=True)





Hello. I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."

In [23]:
#batch
res = model.batch(["hi who are you?","what do you do?","what is your cutoff date?"],config={"max_concurrency":5})

for chunk in res :
    print(chunk.content)



Hello. I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."
I can be used in a variety of ways, from helping you plan a vacation to creating art. I'm here to assist you in finding the help or information you need. My strengths include answering questions, generating text and images and even just chatting with you.
My knowledge cutoff is currently December 2023, but I have access to more recent information via internet search.


In [29]:
#tools

from langchain.tools import tool

@tool
def get_weather(location : str) -> str:
    """
    get the weather from the location
    """
    return f"the weather at {location} is sunny"

    
model_with_tools = model.bind_tools([get_weather]) #or use create agent

response = model_with_tools.invoke("what is the weather in Chennai?")
print(response)
for tool_call in response.tool_calls:
    print(f"tool :{tool_call['name']}")
    print(f"tool :{tool_call['name']}")



content='' additional_kwargs={'tool_calls': [{'id': 'bh2bjdgsa', 'function': {'arguments': '{"location":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.044091152, 'completion_tokens_details': None, 'prompt_time': 0.011309419, 'prompt_tokens_details': None, 'queue_time': 0.048715422, 'total_time': 0.055400571}, 'model_name': 'llama-3.3-70B-Versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019cba80-902e-7ef0-b4a7-977cdff77900-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Chennai'}, 'id': 'bh2bjdgsa', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234}
tool :get_weather
tool :get_weather


In [ ]:
# #tool execution loop
# 1. USER sends message
#         ↓
# 2. LLM receives message + list of available tools
#         ↓
# 3. LLM DECIDES — do I need a tool?
#         │
#         ├── NO  → generate final answer → END
#         │
#         └── YES → output tool_call (name + args)
#                         ↓
# 4. EXECUTOR reads tool_call
#         ↓
# 5. EXECUTOR runs the actual tool function
#         ↓
# 6. EXECUTOR sends tool RESULT back to LLM
#         ↓
# 7. LLM receives result — go back to step 3
#         ↓
# 8. LLM DECIDES again — need another tool?
#         │
#         ├── YES → repeat steps 4-7
#         │
#         └── NO  → generate final answer → END